# Rerun llama3.3:70b Only

The previous full run produced valid results for `llama3.2` and `mistral`, but `llama3.3:70b` returned `[ERROR]` for all 19,608 rows (likely timeout / VRAM issue).

This notebook:
1. Restores the existing llama3.2 + mistral results from Google Drive
2. Runs a **50-question trial** to verify 70b works before committing to the full run
3. Runs the full 817-question generation for **only** llama3.3:70b
4. Merges the new 70b data with the existing results
5. Re-evaluates, re-plots, and exports the combined dataset

In [ ]:
%cd /content
!rm -rf llm-hallucination-phoenix
!git clone https://github.com/sahelmain/llm-hallucination-phoenix.git
%cd /content/llm-hallucination-phoenix
!git fetch --all
!git reset --hard origin/main
!git clean -fd
!mkdir -p data reports snapshots

import pathlib

p = pathlib.Path("src/run_experiment.py")
if p.exists():
    text = p.read_text()
    text = text.replace("timeout=180", "timeout=600")
    text = text.replace("NUM_WORKERS = 4", "NUM_WORKERS = 1")
    p.write_text(text)
    print("Patched run_experiment.py: timeout -> 600s, NUM_WORKERS -> 1")

!pip install -q -r requirements.txt

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/llm_hallucination_phoenix_checkpoints")
DRIVE_DATA_DIR = DRIVE_CHECKPOINT_ROOT / "data"
DRIVE_REPORTS_DIR = DRIVE_CHECKPOINT_ROOT / "reports"
DRIVE_SNAPSHOT_DIR = DRIVE_CHECKPOINT_ROOT / "snapshots"

for path in [DRIVE_CHECKPOINT_ROOT, DRIVE_DATA_DIR, DRIVE_REPORTS_DIR, DRIVE_SNAPSHOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Drive checkpoint root:", DRIVE_CHECKPOINT_ROOT)

In [ ]:
import shutil
import time
import zipfile
from pathlib import Path

REPO_ROOT = Path.cwd()
LOCAL_SNAPSHOT_DIR = REPO_ROOT / "snapshots"
LOCAL_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)


def _resolve_checkpoint_files(patterns=None, extra_paths=None):
    files = []
    for pattern in patterns or []:
        files.extend(path for path in REPO_ROOT.glob(pattern) if path.is_file())
    for raw_path in extra_paths or []:
        path = Path(raw_path)
        if not path.is_absolute():
            path = REPO_ROOT / path
        if path.exists() and path.is_file():
            files.append(path)
    seen = set()
    deduped = []
    for path in files:
        r = path.resolve()
        if r not in seen:
            deduped.append(path)
            seen.add(r)
    return deduped


def checkpoint_stage(stage_name, patterns=None, extra_paths=None, zip_snapshot=True):
    files_to_save = _resolve_checkpoint_files(patterns=patterns, extra_paths=extra_paths)
    if not files_to_save:
        print(f"No files found for checkpoint stage: {stage_name}")
        return []
    copied = []
    for source in files_to_save:
        relative = source.relative_to(REPO_ROOT)
        destination = DRIVE_CHECKPOINT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)
        copied.append(str(relative))
    if zip_snapshot:
        ts = time.strftime("%Y%m%d_%H%M%S")
        local_zip = LOCAL_SNAPSHOT_DIR / f"{stage_name}_{ts}.zip"
        with zipfile.ZipFile(local_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
            for source in files_to_save:
                archive.write(source, source.relative_to(REPO_ROOT))
        shutil.copy2(local_zip, DRIVE_SNAPSHOT_DIR / local_zip.name)
    print(f"Checkpoint '{stage_name}' saved {len(copied)} files.")
    return copied

In [ ]:
import os
import shutil

for folder_name in ["data", "reports"]:
    source_dir = DRIVE_CHECKPOINT_ROOT / folder_name
    destination_dir = REPO_ROOT / folder_name
    destination_dir.mkdir(parents=True, exist_ok=True)
    restored = 0
    if source_dir.exists():
        for source in source_dir.rglob("*"):
            if source.is_file():
                relative = source.relative_to(DRIVE_CHECKPOINT_ROOT)
                dest = REPO_ROOT / relative
                dest.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, dest)
                restored += 1
    print(f"Restored {restored} files from Drive/{folder_name}")

old_csv = "data/experiment_results.csv"
if not os.path.exists(old_csv):
    old_csv = "data/full_study_results.csv"
if not os.path.exists(old_csv):
    raise FileNotFoundError(
        "No existing results CSV found. Run the main notebook first, "
        "or manually upload experiment_results.csv to data/."
    )

shutil.copy2(old_csv, "data/old_results_backup.csv")
print(f"Backed up {old_csv} -> data/old_results_backup.csv")

import pandas as pd
old_df = pd.read_csv("data/old_results_backup.csv")
print(f"Old data: {len(old_df)} rows")
print("Models:", sorted(old_df["model"].unique().tolist()))
for m in old_df["model"].unique():
    sub = old_df[old_df["model"] == m]
    errs = sub["output"].astype(str).str.startswith("[ERROR]").sum()
    print(f"  {m}: {len(sub)} rows, {errs} errors")

In [ ]:
import os
import subprocess
import time

!curl -fsSL https://ollama.com/install.sh | sh

print("--- GPU check ---")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_FLASH_ATTENTION"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"
os.environ["OLLAMA_KEEP_ALIVE"] = "60m"

ollama_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    env=os.environ.copy(),
)
time.sleep(5)
!ollama --version
print("OLLAMA_NUM_PARALLEL:", os.environ.get("OLLAMA_NUM_PARALLEL"))
print("OLLAMA_FLASH_ATTENTION:", os.environ.get("OLLAMA_FLASH_ATTENTION"))
print("OLLAMA_KEEP_ALIVE:", os.environ.get("OLLAMA_KEEP_ALIVE"))

In [ ]:
!ollama pull llama3.3:70b
!ollama pull mistral
!ollama list

In [ ]:
import time
from datasets import load_dataset
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
ds = load_dataset("truthful_qa", "generation", split="validation")
trial_df = ds.to_pandas().sample(n=50, random_state=42).reset_index(drop=True)

errors = 0
successes = 0
start = time.time()

for i, row in trial_df.iterrows():
    try:
        resp = client.chat.completions.create(
            model="llama3.3:70b",
            messages=[{"role": "user", "content": f"Answer concisely: {row['question']}"}],
            temperature=0.0,
            max_tokens=128,
            timeout=600,
        )
        output = resp.choices[0].message.content.strip()
        if output:
            successes += 1
            if i < 5:
                print(f"[{i+1}/50] OK: {output[:100]}")
            elif i % 10 == 0:
                print(f"[{i+1}/50] OK ({successes} good so far)")
        else:
            errors += 1
            print(f"[{i+1}/50] EMPTY response")
    except Exception as e:
        errors += 1
        print(f"[{i+1}/50] ERROR: {e}")

elapsed = time.time() - start
error_rate = errors / 50
sec_per_q = elapsed / 50
full_calls = 817 * 4 * 2 * 3
est_hours = sec_per_q * full_calls / 3600

print(f"\n{'='*50}")
print(f"TRIAL RESULTS")
print(f"{'='*50}")
print(f"Successes : {successes}/50")
print(f"Errors    : {errors}/50 ({error_rate:.0%})")
print(f"Time      : {elapsed:.0f}s ({sec_per_q:.1f}s per question)")
print(f"")
print(f"Estimated full run (3 reps): {est_hours:.1f} hours ({full_calls} calls)")
print(f"Estimated full run (1 rep) : {est_hours/3:.1f} hours ({full_calls//3} calls)")

if error_rate > 0.2:
    raise RuntimeError(
        f"Trial FAILED: {error_rate:.0%} error rate. "
        f"The 70b model likely does not fit in GPU memory. "
        f"Try a stronger GPU runtime (A100) or a smaller large model."
    )
print("\nTrial PASSED. Proceed to full run.")

In [ ]:
# Set REPS to match the existing data (3) or reduce to 1 for a faster run.
# If you set REPS = 1, the merge step will also trim old data to rep=0.
REPS = 3
print("REPS =", REPS)

In [ ]:
import yaml

with open("config/experiment.yaml") as f:
    cfg = yaml.safe_load(f)

cfg["models"]["experiment_models"] = [
    {"provider": "ollama", "model_name": "llama3.3:70b", "temperature": 0.0, "max_tokens": 256},
]
cfg["prompt_templates"] = [
    "factual_direct", "strict_abstain", "chain_of_thought", "concise_factual",
]
cfg["dataset"]["sample_size"] = 817
cfg["evaluation"]["repetitions_per_item"] = REPS
cfg.setdefault("execution", {}).setdefault("full_study", {})["max_workers"] = 1

with open("config/experiment.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

n_calls = 817 * 4 * 2 * REPS
print(f"Model: llama3.3:70b only")
print(f"Templates: {cfg['prompt_templates']}")
print(f"Reps: {REPS}")
print(f"Total calls: {n_calls}")

In [ ]:
import os
import subprocess
import time

RUN_CMD = "python -u src/run_experiment.py"
print("Executing:", RUN_CMD)

checkpoint_interval = 300
last_ckpt = 0.0

proc = subprocess.Popen(
    RUN_CMD, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
    now = time.time()
    if now - last_ckpt >= checkpoint_interval:
        partial = [p for p in ["data/experiment_results.csv"] if os.path.exists(p)]
        if partial:
            checkpoint_stage("70b_generation_partial", extra_paths=partial)
            last_ckpt = now
            print("\nCheckpointed partial 70b outputs to Drive.")

ret = proc.wait()
if ret != 0:
    raise RuntimeError(f"Generation failed with exit code {ret}")

import pandas as pd
new_df = pd.read_csv("data/experiment_results.csv")
errs = new_df["output"].astype(str).str.startswith("[ERROR]").sum()
print(f"\n70b generation complete: {len(new_df)} rows, {errs} errors ({errs/len(new_df):.1%})")

new_df.to_csv("data/70b_only_results.csv", index=False)
checkpoint_stage("70b_generation_complete", extra_paths=["data/70b_only_results.csv"])
print("Saved data/70b_only_results.csv")

In [ ]:
import pandas as pd

old_df = pd.read_csv("data/old_results_backup.csv")
new_70b = pd.read_csv("data/70b_only_results.csv")

if REPS == 1 and old_df["repetition"].nunique() > 1:
    old_df = old_df[old_df["repetition"] == 0].reset_index(drop=True)
    print(f"Trimmed old data to rep=0: {len(old_df)} rows")

keep_df = old_df[old_df["model"] != "llama3.3:70b"].reset_index(drop=True)
print(f"Old data without 70b: {len(keep_df)} rows")
print(f"New 70b data: {len(new_70b)} rows")

merged = pd.concat([keep_df, new_70b], ignore_index=True)
merged.to_csv("data/experiment_results.csv", index=False)
merged.to_csv("data/full_study_results.csv", index=False)

print(f"\nMerged: {len(merged)} rows")
for m in sorted(merged["model"].unique()):
    sub = merged[merged["model"] == m]
    errs = sub["output"].astype(str).str.startswith("[ERROR]").sum()
    print(f"  {m}: {len(sub)} rows, {errs} errors")

checkpoint_stage("merged_results", extra_paths=["data/experiment_results.csv", "data/full_study_results.csv"])

In [ ]:
import os
import subprocess
import yaml

with open("config/experiment.yaml") as f:
    eval_cfg = yaml.safe_load(f)

eval_cfg["models"]["experiment_models"] = [
    {"provider": "ollama", "model_name": "llama3.2", "temperature": 0.0, "max_tokens": 128},
    {"provider": "ollama", "model_name": "mistral", "temperature": 0.0, "max_tokens": 128},
    {"provider": "ollama", "model_name": "llama3.3:70b", "temperature": 0.0, "max_tokens": 256},
]
if "judge_model" not in eval_cfg.get("models", {}):
    eval_cfg.setdefault("models", {})["judge_model"] = {
        "provider": "ollama", "model_name": "mistral",
        "temperature": 0.0, "max_tokens": 16,
    }
if "evaluators" not in eval_cfg:
    eval_cfg["evaluators"] = {}

with open("config/experiment.yaml", "w") as f:
    yaml.dump(eval_cfg, f, default_flow_style=False)

import pathlib
p = pathlib.Path("src/evaluate_metrics.py")
if p.exists():
    text = p.read_text()
    if "NUM_WORKERS = 1" in text:
        p.write_text(text.replace("NUM_WORKERS = 1", "NUM_WORKERS = 4"))
    elif "NUM_WORKERS = 4" not in text and "NUM_WORKERS =" in text:
        pass

subprocess.run("python -u src/evaluate_metrics.py", shell=True, check=True)

eval_outputs = [
    "data/evaluated_results.csv", "data/metrics.json",
]
checkpoint_stage("post_evaluation", extra_paths=eval_outputs)
print("Evaluation complete and checkpointed.")

In [ ]:
import subprocess

subprocess.run("python -u src/generate_plots.py", shell=True, check=True)

if os.path.isfile("src/generate_full_study_artifacts.py"):
    subprocess.run("python -u src/generate_full_study_artifacts.py --suffix full", shell=True, check=True)

checkpoint_stage(
    "post_plots",
    patterns=["data/*.png", "data/table_*", "data/category_*", "data/paired_*", "reports/*.md"],
    extra_paths=["data/metrics.json", "data/evaluated_results.csv"],
)
print("Plots and artifacts checkpointed.")

In [ ]:
import glob
import json
import pandas as pd

df = pd.read_csv("data/experiment_results.csv")
with open("data/metrics.json") as f:
    metrics = json.load(f)

error_count = int(df["output"].astype(str).str.startswith("[ERROR]").sum())
pngs = sorted(glob.glob("data/*.png"))

print("Verification")
print("rows:", len(df))
print("models:", sorted(df["model"].unique().tolist()))
print("templates:", sorted(df["template"].unique().tolist()))
print("reps:", sorted(df["repetition"].unique().tolist()))
print("errors:", error_count, f"({error_count/len(df):.1%})")
print("hallucination_rate:", round(metrics["hallucination_rate"]["value"], 4))
print("accuracy:", round(metrics["accuracy"]["value"], 4))
print("plot_count:", len(pngs))
print()
print("Per model:")
for m, v in metrics.get("by_model", {}).items():
    print(f"  {m}: HR={v['hallucination_rate']['value']:.3f}, Acc={v['accuracy']['value']:.3f}")

if error_count / len(df) > 0.05:
    print(f"\nWARNING: {error_count/len(df):.1%} error rate is high. Check 70b outputs.")
else:
    print("\nVerification PASSED.")

In [ ]:
import os
import shutil
import time
from google.colab import files

checkpoint_stage("final_bundle", patterns=["data/*", "reports/*"])

!zip -r full_study_outputs.zip data/ reports/
!ls -lh full_study_outputs.zip

bundle_name = f"full_study_outputs_{time.strftime('%Y%m%d_%H%M%S')}.zip"
shutil.copy2("full_study_outputs.zip", DRIVE_SNAPSHOT_DIR / bundle_name)
print("Saved to Drive:", DRIVE_SNAPSHOT_DIR / bundle_name)

files.download("full_study_outputs.zip")